# Kalori Harcamasını Tahmin Edin

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import root_mean_squared_error
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [2]:
train= pd.read_csv('train(3).csv')
test= pd.read_csv('test(3).csv')

In [3]:
train.head()

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


In [4]:
test.head()

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,750000,male,45,177.0,81.0,7.0,87.0,39.8
1,750001,male,26,200.0,97.0,20.0,101.0,40.5
2,750002,female,29,188.0,85.0,16.0,102.0,40.4
3,750003,female,39,172.0,73.0,20.0,107.0,40.6
4,750004,female,30,173.0,67.0,16.0,94.0,40.5


In [5]:
HEDEF = 'Calories'

In [6]:
# Cinsiyet Sütununu Güvenle Sayısallaştırma
#Sex (Male/Female) sütununu OrdinalEncoder ile 0 ve 1 değerlerine dönüştürmek.
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[['Sex']] = encoder.fit_transform(train[['Sex']].astype(str))
test[['Sex']] = encoder.transform(test[['Sex']].astype(str))

In [7]:
# Sağlık ve Spor Bilimi Özellik Mühendisliği (BMR & Yoğunluk Formülleri)
#Mifflin-St Jeor BMR Formülü: Cinsiyete bağlı olarak kişinin bazal metabolizma hızını hesaplayan yeni bir sütun üretmek.
#Kalp Ritmi Yoğunluğu (HR Intensity): Kalp ritminin aktivite süresine oranı (Heart_Rate * Duration).
#Metabolik Isı İndeksi: Vücut sıcaklığı ile egzersiz süresinin çarpımı (Body_Temp * Duration)
for df in [train, test]:
    # Mifflin-St Jeor BMR Formülü (Erkek ve Kadın için katsayılar farklıdır)
    # Kadınlar için (Sex == 0 varsayılırsa veya tam tersi, formül genel eğilimi yakalar)
    df['bmr'] = (10 * df['Weight']) + (6.25 * df['Height']) - (5 * df['Age'])
    df.loc[df['Sex'] == 1, 'bmr'] += 5
    df.loc[df['Sex'] == 0, 'bmr'] -= 161
    
    # Egzersiz Yoğunluk İndikatörleri (Şampiyon kombinasyonları)
    df['hr_duration_impact'] = df['Heart_Rate'] * df['Duration']
    df['temp_duration_impact'] = df['Body_Temp'] * df['Duration']
    df['weight_duration_ratio'] = df['Weight'] / (df['Duration'] + 1e-5)

print("2. Adım: Fizyolojik özellik mühendisliği tamamlandı. Modelleme başlatılıyor...")


2. Adım: Fizyolojik özellik mühendisliği tamamlandı. Modelleme başlatılıyor...


In [8]:
# Girdileri (X) ve Hedefi (y) Kesin Olarak Ayırma
giris_sutunlari = [kolon for kolon in train.columns if kolon not in ['id', HEDEF]]

X = train[giris_sutunlari]
# RMSLE metriği için hedef değişkenin logaritmasını (log1p) alıyoruz!
y = np.log1p(train[HEDEF]) 
X_test = test[giris_sutunlari]

In [9]:
# 5. 5-Fold Cross Validation Düzeni
#Sürekli sayısal veriler için kararlı ve hızlı çalışan LightGBM modelini kurgulamak.
#Veriyi 5 parçaya bölerek (5-Fold Cross-Validation) eğitimi tamamlamak.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_tahminleri = np.zeros(len(train))
test_tahminleri = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Regresyon için LightGBM kurulumu
    model = LGBMRegressor(
        n_estimators=600,
        learning_rate=0.04,
        random_state=42,
        verbose=-1
    )
    model.fit(X_train, y_train)
    
    # Katman tahminlerini toplama
    oof_tahminleri[val_idx] = model.predict(X_val)
    test_predictions_fold = model.predict(X_test)
    
    # Negatif tahmin olasılıklarını RMSLE çökmemesi için sıfıra eşitliyoruz
    test_predictions_fold = np.clip(test_predictions_fold, 0, None)
    test_tahminleri += test_predictions_fold / kf.n_splits
    
    # Bu katmanın RMSLE skorunu hesaplayalım (Hedef zaten logaritmalı olduğu için standart RMSE bakıyoruz)
    fold_rmsle = root_mean_squared_error(y_val, oof_tahminleri[val_idx])
    print(f"Katman {fold + 1} RMSLE Skoru: {fold_rmsle:.4f}")

# Genel Başarı Oranı
genel_rmsle = root_mean_squared_error(y, oof_tahminleri)
print(f"\n---> TÜM VERİ SETİ GENEL RMSLE SKORU: {genel_rmsle:.4f} <---")


Katman 1 RMSLE Skoru: 0.0602
Katman 2 RMSLE Skoru: 0.0610
Katman 3 RMSLE Skoru: 0.0604
Katman 4 RMSLE Skoru: 0.0608
Katman 5 RMSLE Skoru: 0.0602

---> TÜM VERİ SETİ GENEL RMSLE SKORU: 0.0605 <---


In [10]:
#  Kaggle Gönderi Dosyasının Oluşturulması (Logaritmayı geri çeviriyoruz)
# np.expm1 fonksiyonu, np.log1p işleminin tam tersini yaparak orijinal kalori değerlerini bulur
nihai_kalori_tahminleri = np.expm1(test_tahminleri)

submission = pd.DataFrame({
    'id': test['id'],
    'Calories': nihai_kalori_tahminleri
})

In [11]:
submission.to_csv('submission_calories.csv', index=False)